# Session 8 — Your turn: Computer vision: why images need a different network

Two jobs about convolutions: predict the shapes and parameter counts before running the network, and write the sliding-filter loop yourself.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np
import torch
from torch import nn
from scipy.signal import correlate2d
from sklearn.datasets import load_digits

cnn = nn.Sequential(nn.Conv2d(1, 16, 3), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(800, 10))
print(sum(p.numel() for p in cnn.parameters()), "parameters in the CNN of the session")
from aij_check import check, close, summary

12810 parameters in the CNN of the session


## Task 1: Predict the shapes and the parameters

Write `conv_out(size, kernel, stride=1, padding=0)` (the output size of a convolution along one side) and `conv_params(c_in, c_out, kernel)` (weights plus one bias per filter).
Use them with `pool_out = size // 2` to fill `after_conv1`, `after_pool1`, `after_conv2`, `after_pool2` (the side length of the square maps, starting from a 28 x 28 image),
and `params_conv1`, `params_conv2`. Then `params_total` adds the final linear layer: `32 * after_pool2 * after_pool2` inputs to 10 outputs, with biases.

In [2]:
def conv_out(size, kernel, stride=1, padding=0):
    return (size + 2 * padding - kernel) // stride + 1


def conv_params(c_in, c_out, kernel):
    return c_out * (c_in * kernel * kernel + 1)


after_conv1 = conv_out(28, 3)
after_pool1 = after_conv1 // 2
after_conv2 = conv_out(after_pool1, 3)
after_pool2 = after_conv2 // 2
params_conv1 = conv_params(1, 16, 3)
params_conv2 = conv_params(16, 32, 3)
params_total = params_conv1 + params_conv2 + (32 * after_pool2 * after_pool2 * 10 + 10)
print(after_conv1, after_pool1, after_conv2, after_pool2, params_conv1, params_conv2, params_total)

26 13 11 5 160 4640 12810


In [3]:
check("the side lengths are 26, 13, 11, 5", lambda: (after_conv1, after_pool1, after_conv2, after_pool2) == (26, 13, 11, 5))
check("the first layer has 160 parameters, the second 4,640", lambda: (params_conv1, params_conv2) == (160, 4640))
check("the total matches PyTorch's own count", lambda: params_total == sum(p.numel() for p in cnn.parameters()), "Do not forget the 10 biases of the linear layer.")
check("conv_out handles padding and stride", lambda: conv_out(28, 3, stride=2, padding=1) == 14)
summary()

PASS  the side lengths are 26, 13, 11, 5
PASS  the first layer has 160 parameters, the second 4,640
PASS  the total matches PyTorch's own count
PASS  conv_out handles padding and stride

4 of 4 checks passed.  Well done.


**Why it works.** A filter is shared across the whole image, so the parameter count depends on the filter size and the number of channels, not on the image size. That is why 12,810 parameters are enough where a dense layer on 784 pixels needs 100,480 for its first layer alone.

## Task 2: Slide a filter yourself

Write `convolve(image, kernel)` that slides the kernel over the image with no padding (the 'valid' region), multiplying and adding at every position (as in the session, it does not flip the kernel).
Apply a horizontal-edge filter `[[-1,-1,-1],[0,0,0],[1,1,1]]` to the first digit of `load_digits()` and store the position of its strongest absolute response in `strongest`, as `(row, column)`.

In [4]:
def convolve(image, kernel):
    kh, kw = kernel.shape
    out = np.zeros((image.shape[0] - kh + 1, image.shape[1] - kw + 1))
    for r in range(out.shape[0]):
        for c in range(out.shape[1]):
            out[r, c] = np.sum(image[r:r + kh, c:c + kw] * kernel)
    return out


image = load_digits().images[0] / 16.0
horizontal = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=float)
response = convolve(image, horizontal)
strongest = tuple(int(i) for i in np.unravel_index(np.abs(response).argmax(), response.shape))
print(response.shape, strongest)

(6, 6) (1, 3)


In [5]:
check("the output is 6 x 6 for an 8 x 8 image and a 3 x 3 filter", lambda: np.shape(response) == (6, 6))
check("it agrees with scipy's correlate2d", lambda: close(response, correlate2d(image, horizontal, mode="valid"), 1e-9))
check("strongest is the position of the largest absolute response", lambda: strongest == tuple(int(i) for i in np.unravel_index(np.abs(correlate2d(image, horizontal, mode="valid")).argmax(), (6, 6))))
summary()

PASS  the output is 6 x 6 for an 8 x 8 image and a 3 x 3 filter
PASS  it agrees with scipy's correlate2d
PASS  strongest is the position of the largest absolute response

3 of 3 checks passed.  Well done.


**Why it works.** The nine multiplications and one sum at each position are the whole operation; a CNN learns the nine numbers instead of fixing them. The response map has the size conv_out predicted: 8 - 3 + 1 = 6.